# Preprocesamiento: Pipeline de Producción
Este notebook crea un **pipeline completo de sklearn** para el preprocesamiento del conjunto de datos de precios de viviendas de King County. El pipeline encapsula todas las transformaciones, permitiendo:

1. **Inferencia de extremo a extremo**: Transformar datos CSV en bruto directamente
2. **Reproducibilidad**: Todos los parámetros ajustados guardados en un único artefacto
3. **Sin fuga de datos**: Separación correcta de ajuste/transformación garantizada por diseño

> **Prerrequisito:** Para explicaciones paso a paso de cada etapa del preprocesamiento, consulta `04a-preprocessing-step-by-step.ipynb`.

In [ ]:
import numpy as npimport pandas as pdfrom pathlib import Path

## 1. Carga y Preparación de Datos

In [ ]:
import kagglehubpath = kagglehub.dataset_download("harlfoxem/housesalesprediction")csv_path = Path(path) / "kc_house_data.csv"df = pd.read_csv(csv_path)print(f"Dataset shape: {df.shape}")

Dataset shape: (21613, 21)


### Parseo de fechas para la división
Parseamos las fechas aquí para permitir la división temporal. El pipeline manejará su propio parseo de fechas para la inferencia de extremo a extremo.

In [ ]:
df["date_parsed"] = pd.to_datetime(df["date"].str[:8], format="%Y%m%d")df = df.sort_values("date_parsed").reset_index(drop=True)print(f"Date range: {df['date_parsed'].min().date()} to {df['date_parsed'].max().date()}")

Date range: 2014-05-02 to 2015-05-27


### División temporal

In [ ]:
def temporal_train_val_test_split(    df: pd.DataFrame,     date_column: str,    val_size: float = 0.15,     test_size: float = 0.15) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:    """Split data temporally into train, validation, and test sets."""    df_sorted = df.sort_values(date_column).reset_index(drop=True)    n = len(df_sorted)    train_end = int(n * (1 - val_size - test_size))    val_end = int(n * (1 - test_size))    return (        df_sorted.iloc[:train_end].copy(),        df_sorted.iloc[train_end:val_end].copy(),        df_sorted.iloc[val_end:].copy()    )train_df, val_df, test_df = temporal_train_val_test_split(df, date_column="date_parsed")print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")print(f"Train dates: {train_df['date_parsed'].min().date()} to {train_df['date_parsed'].max().date()}")print(f"Val dates:   {val_df['date_parsed'].min().date()} to {val_df['date_parsed'].max().date()}")print(f"Test dates:  {test_df['date_parsed'].min().date()} to {test_df['date_parsed'].max().date()}")

Train: 15,129 | Val: 3,242 | Test: 3,242
Train dates: 2014-05-02 to 2015-01-16
Val dates:   2015-01-16 to 2015-03-26
Test dates:  2015-03-26 to 2015-05-27


## 2. Transformadores Personalizados
Creamos transformadores sklearn personalizados para encapsular toda la lógica de preprocesamiento. Cada transformador sigue el protocolo de sklearn:
- `fit(X, y=None)`: Aprender parámetros solo de los datos de entrenamiento
- `transform(X)`: Aplicar transformación usando los parámetros aprendidos
- Los parámetros ajustados terminan con guion bajo (p. ej., `min_date_`)

### 2.1 DateParser
Transforma la cadena de fecha en bruto en una columna datetime. Esto permite la inferencia de extremo a extremo desde datos CSV.

Este es un transformador **sin estado** (no necesita ajuste) — simplemente parsea cadenas, por lo que podemos definirlo con `FunctionTransformer`.

In [ ]:
import pandas as pdfrom sklearn.preprocessing import FunctionTransformerdef parse_date_strings(X: pd.DataFrame, date_column: str = "date", output_column: str = "date_parsed") -> pd.DataFrame:    X = X.copy()    X[output_column] = pd.to_datetime(X[date_column].str[:8], format="%Y%m%d")    return X# Create the transformer instance to use in your pipelinedate_parser = FunctionTransformer(    func=parse_date_strings,     kw_args={"date_column": "date", "output_column": "date_parsed"})

### 2.2 FeatureEngineer
Crea todas las características derivadas y elimina las columnas no predictivas.

> **Diseño clave:** La `min_date_` se aprende durante `fit()` de los datos de entrenamiento, luego se usa en `transform()` para todos los datos. Esto previene la fuga de datos en la característica `days_since_start`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixinclass FeatureEngineer(BaseEstimator, TransformerMixin):    """    Feature engineering transformer for house price prediction.        Creates derived features and drops non-predictive columns.        Fitted Parameters    -----------------    min_date_ : Timestamp        Minimum date from training data. Used to compute days_since_start        consistently across all splits (preventing data leakage).        Features Created    ----------------    - days_since_start: Days since min_date_ (temporal trend)    - house_age: Age of house at sale time    - was_renovated: Binary indicator    - years_since_renovation: Years since last renovation    - basement_ratio: sqft_basement / sqft_living    - living_vs_neighbors: sqft_living / sqft_living15    - lot_vs_neighbors: sqft_lot / sqft_lot15        Columns Dropped    ---------------    id, date, date_parsed, zipcode, yr_built, yr_renovated    """        def __init__(self):        pass        def fit(self, X: pd.DataFrame, y=None):        """        Learn reference values from training data.                Stores min_date_ to ensure consistent days_since_start across splits.        """        # Validate required columns        required = ["date_parsed", "yr_built", "yr_renovated", "sqft_basement",                     "sqft_living", "sqft_living15", "sqft_lot", "sqft_lot15"]        missing = set(required) - set(X.columns)        if missing:            raise ValueError(f"Missing required columns: {missing}")                # Store minimum date from entrenamiento datos to calculate days_since_start        self.min_date_ = X["date_parsed"].min()                return self        def transform(self, X: pd.DataFrame) -> pd.DataFrame:        """        Apply feature engineering transformations.                Uses fitted min_date_ for days_since_start calculation.        """        X = X.copy()                # Temporal característica: days since entrenamiento start        # Uses fitted min_date_ (NOT recomputed from X)        X["days_since_start"] = (X["date_parsed"] - self.min_date_).dt.days                # Age características        sale_year = X["date_parsed"].dt.year        X["house_age"] = sale_year - X["yr_built"]        X["was_renovated"] = (X["yr_renovated"] > 0).astype(int)        X["years_since_renovation"] = np.where(            X["yr_renovated"] > 0,            sale_year - X["yr_renovated"],            0        )                # Ratio características        X["basement_ratio"] = X["sqft_basement"] / X["sqft_living"].replace(0, 1)        X["living_vs_neighbors"] = X["sqft_living"] / X["sqft_living15"].replace(0, 1)        X["lot_vs_neighbors"] = X["sqft_lot"] / X["sqft_lot15"].replace(0, 1)                # Drop non-predictive columns        columns_to_drop = [            "id",           # Property identifier            "date",         # Original cadena format            "date_parsed",  # Used for ingeniería de características only            "zipcode",      # High cardinality (using lat/long)            "yr_built",     # Replaced by house_age            "yr_renovated", # Replaced by was_renovated, years_since_renovation        ]        X = X.drop(columns=columns_to_drop)                return X

## 3. Construcción del Pipeline Completo
El pipeline tiene tres etapas:
1. **DateParser**: Cadena de fecha en bruto → datetime
2. **FeatureEngineer**: Crear características derivadas, eliminar columnas no predictivas
3. **NumericPreprocessor**: Transformación logarítmica + escalado

In [ ]:
from sklearn.pipeline import Pipelinefrom sklearn.compose import ColumnTransformerfrom sklearn.preprocessing import StandardScaler, FunctionTransformer

In [ ]:
# Define característica groups (after ingeniería de características)log_features = [    "sqft_living", "sqft_lot", "sqft_above", "sqft_basement",    "sqft_living15", "sqft_lot15"]passthrough_features = ["waterfront", "was_renovated"]# Log transformation + scaling pipelinelog_pipeline = Pipeline([    ("log", FunctionTransformer(np.log1p, validate=True, feature_names_out="one-to-one")),    ("scale", StandardScaler())])# Numeric preprocessor (applied after ingeniería de características)numeric_preprocessor = ColumnTransformer([    ("log", log_pipeline, log_features),    ("passthrough", "passthrough", passthrough_features)], remainder=StandardScaler()) # Automatically scales all remaining columns# COMPLETE PIPELINE: Raw datos → Processed característicasfull_pipeline = Pipeline([    ("date_parser", date_parser),    ("feature_engineer", FeatureEngineer()),    ("preprocessing", numeric_preprocessor)])full_pipeline

/media/DIURNOext4/alejandro/wip-clase/.venv/lib/python3.11/site-packages/sklearn/externals/_numpydoc/docscrape.py:420: UserWarning: Unknown section Fitted Parameters
  self[section] = content
/media/DIURNOext4/alejandro/wip-clase/.venv/lib/python3.11/site-packages/sklearn/externals/_numpydoc/docscrape.py:420: UserWarning: Unknown section Features Created
  self[section] = content
/media/DIURNOext4/alejandro/wip-clase/.venv/lib/python3.11/site-packages/sklearn/externals/_numpydoc/docscrape.py:420: UserWarning: Unknown section Columns Dropped
  self[section] = content


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('date_parser', ...), ('feature_engineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function par...x728ca20cb880>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arg

## 4. Ajuste y Transformación
Ajustamos el pipeline completo con los datos de entrenamiento y luego transformamos todas las divisiones.

In [ ]:
# Prepare raw X (without date_parsed, but with date cadena)X_train = train_df.drop(columns=["price", "date_parsed"])y_train = train_df["price"]X_val = val_df.drop(columns=["price", "date_parsed"])y_val = val_df["price"]X_test = test_df.drop(columns=["price", "date_parsed"])y_test = test_df["price"]print(f"Raw X shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

Raw X shapes: Train=(15129, 20), Val=(3242, 20), Test=(3242, 20)


In [ ]:
# FIT on entrenamiento datos onlyfull_pipeline.fit(X_train, y_train)# Show fitted parametersfitted_min_date = full_pipeline.named_steps['feature_engineer'].min_date_print(f"Fitted min_date from training: {fitted_min_date.date()}")

Fitted min_date from training: 2014-05-02


In [ ]:
# Transform all splitsX_train_processed = full_pipeline.transform(X_train)X_val_processed = full_pipeline.transform(X_val)X_test_processed = full_pipeline.transform(X_test)print(f"Processed shapes:")print(f"  Train: {X_train_processed.shape}")print(f"  Val:   {X_val_processed.shape}")print(f"  Test:  {X_test_processed.shape}")

Processed shapes:
  Train: (15129, 22)
  Val:   (3242, 22)
  Test:  (3242, 22)


In [ ]:
# Get característica namesfeature_names = full_pipeline.named_steps['preprocessing'].get_feature_names_out()print(f"\nTotal features: {len(feature_names)}")print(f"Feature names: {list(feature_names)}")


Total features: 22
Feature names: ['log__sqft_living', 'log__sqft_lot', 'log__sqft_above', 'log__sqft_basement', 'log__sqft_living15', 'log__sqft_lot15', 'passthrough__waterfront', 'passthrough__was_renovated', 'remainder__bedrooms', 'remainder__bathrooms', 'remainder__floors', 'remainder__view', 'remainder__condition', 'remainder__grade', 'remainder__lat', 'remainder__long', 'remainder__days_since_start', 'remainder__house_age', 'remainder__years_since_renovation', 'remainder__basement_ratio', 'remainder__living_vs_neighbors', 'remainder__lot_vs_neighbors']


## 5. Validación del Pipeline
Verificamos que el pipeline previene correctamente la fuga de datos y funciona de extremo a extremo.

### 5.1 Verificar la consistencia de `days_since_start`
La característica `days_since_start` debe tener **rangos crecientes** entre divisiones (dado que están ordenadas temporalmente).

In [ ]:
# Find the índice of days_since_start in the final característicasdays_idx = list(feature_names).index('remainder__days_since_start')train_days = X_train_processed[:, days_idx]val_days = X_val_processed[:, days_idx]test_days = X_test_processed[:, days_idx]print("days_since_start (SCALED) ranges:")print(f"  Train: {train_days.min():.2f} to {train_days.max():.2f}")print(f"  Val:   {val_days.min():.2f} to {val_days.max():.2f}")print(f"  Test:  {test_days.min():.2f} to {test_days.max():.2f}")# validación: val min should be > train min (it's later in time)assert val_days.min() > train_days.min()assert test_days.min() > val_days.min()

days_since_start (SCALED) ranges:
  Train: -1.67 to 1.99
  Val:   1.99 to 2.97
  Test:  2.97 to 3.85


### 5.2 Prueba de inferencia de extremo a extremo
El pipeline debe poder transformar datos completamente en bruto (como vendrían de un CSV) sin ningún preprocesamiento previo.

In [ ]:
# Create a synthetic "new" fila as it would come from raw CSVnew_data = pd.DataFrame([{    "id": 9999999,    "date": "20150701T000000",  # Raw date cadena!    "bedrooms": 3,    "bathrooms": 2.0,    "sqft_living": 1800,    "sqft_lot": 5000,    "floors": 1.0,    "waterfront": 0,    "view": 0,    "condition": 3,    "grade": 7,    "sqft_above": 1800,    "sqft_basement": 0,    "yr_built": 1990,    "yr_renovated": 0,    "zipcode": 98001,    "lat": 47.5,    "long": -122.2,    "sqft_living15": 1750,    "sqft_lot15": 4800}])print("Raw input data:")print(new_data[["date", "sqft_living", "yr_built"]].to_string(index=False))

Raw input data:
           date  sqft_living  yr_built
20150701T000000         1800      1990


In [ ]:
# Transform using the complete pipelinenew_processed = full_pipeline.transform(new_data)print(f"Processed shape: {new_processed.shape}")print(f"Processed values (first 5 features):")for i, name in enumerate(feature_names[:5]):    print(f"  {name}: {new_processed[0, i]:.4f}")print("\n✓ End-to-end inference works!")

Processed shape: (1, 22)
Processed values (first 5 features):
  log__sqft_living: -0.1486
  log__sqft_lot: -0.5295
  log__sqft_above: 0.2176
  log__sqft_basement: -0.8008
  log__sqft_living15: -0.2343

✓ End-to-end inference works!


## 6. Guardar Pipeline y Datos

In [ ]:
import jobliboutput_dir = Path("processed_data")output_dir.mkdir(exist_ok=True)# Save processed arraysnp.save(output_dir / "X_train.npy", X_train_processed)np.save(output_dir / "X_val.npy", X_val_processed)np.save(output_dir / "X_test.npy", X_test_processed)np.save(output_dir / "y_train.npy", y_train.values)np.save(output_dir / "y_val.npy", y_val.values)np.save(output_dir / "y_test.npy", y_test.values)# Save the COMPLETE pipeline (includes everything!)joblib.dump(full_pipeline, output_dir / "preprocessor.joblib")# Save característica namesnp.save(output_dir / "feature_names.npy", feature_names)print(f"Saved to {output_dir.absolute()}")

Saved to /media/DIURNOext4/alejandro/wip-clase/PIA-SAA/example_repos/king-county/processed_data


## 7. Ejemplo de Inferencia
Esta sección muestra cómo usar el pipeline guardado para inferencia con nuevos datos.

In [ ]:
# Simulate loading in a new sessionloaded_pipeline = joblib.load(output_dir / "preprocessor.joblib")# Check that fitted parameters are preservedprint(f"Loaded pipeline min_date: {loaded_pipeline.named_steps['feature_engineer'].min_date_.date()}")

Loaded pipeline min_date: 2014-05-02


In [ ]:
# Process new datos using loaded pipelinenew_data_inference = pd.DataFrame([{    "id": 8888888,    "date": "20151015T000000",    "bedrooms": 4,    "bathrooms": 2.5,    "sqft_living": 2500,    "sqft_lot": 8000,    "floors": 2.0,    "waterfront": 0,    "view": 1,    "condition": 4,    "grade": 8,    "sqft_above": 2000,    "sqft_basement": 500,    "yr_built": 2005,    "yr_renovated": 0,    "zipcode": 98052,    "lat": 47.6,    "long": -122.1,    "sqft_living15": 2400,    "sqft_lot15": 7500}])X_new = loaded_pipeline.transform(new_data_inference)print(f"Inference result shape: {X_new.shape}")print("\n✓ Ready for model prediction!")

Inference result shape: (1, 22)

✓ Ready for model prediction!


## Resumen

### Contenido del Pipeline
El archivo `preprocessor.joblib` guardado contiene:

| Etapa | Transformador | Parámetros Ajustados |
|-------|-------------|-------------------|
| 1 | DateParser | Ninguno (sin estado) |
| 2 | FeatureEngineer | `min_date_` (del entrenamiento) |
| 3 | ColumnTransformer | Medias/desviaciones del escalador |

### Cómo usar para inferencia
```python
import joblib
import pandas as pd

# Cargar pipeline
pipeline = joblib.load("processed_data/preprocessor.joblib")

# Cargar datos CSV en bruto
new_data = pd.read_csv("new_houses.csv")

# Transformar (maneja todo automáticamente)
X = pipeline.transform(new_data)

# Predecir (requiere modelo entrenado)
model = joblib.load("model.joblib")
predictions = model.predict(X)
```

### Ventajas clave
- ✅ **Inferencia de extremo a extremo**: Acepta datos CSV en bruto con cadenas de fecha
- ✅ **Artefacto único**: Todo el preprocesamiento en un archivo `joblib`
- ✅ **Compatible con sklearn**: Funciona con `GridSearchCV`, `cross_val_score`, etc.